# Data Import Manager

Builds the UTC-merged profile + day-ahead price dataset for a site, replacing the old per-profile `*_to_CSV.ipynb` + `csvmerger.ipynb` pair with a single notebook that calls reusable functions from `srce/data_import.py`.

Pipeline per profile:
1. Read the site's Excel profile (`con`/`gen`/`off`/`inj`), localize its naive timestamps to the source timezone, convert to UTC.
2. Read the market data CSV (already UTC).
3. Inner-join both on the UTC timestamp.
4. Save the result to `data/csvs/`, in the same column layout as the existing `merged*ANDda_belgium.csv` files.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent))
from srce.data_import import PROFILES, build_profile_dataset

DATA_DIR = Path.cwd().parent / "data"
print("Known profiles:", list(PROFILES))

Known profiles: ['carmeuse', 'montea', 'lemahieu']


## Choose a profile and build its merged dataset

Set `PROFILE_NAME` to one of the keys printed above, then run the cell below. `source_tz` defaults to fixed CET (UTC+1, no DST) to match the current pipeline -- only override it if a profile's source timestamps are known to be real DST-aware local time.

In [2]:
PROFILE_NAME = "montea"  # one of: "carmeuse", "montea", "lemahieu"
MARKET_FILENAME = "da_belgium.csv"
OUTPUT_FILENAME = f"merged{PROFILE_NAME}ANDda_belgium.csv"

merged = build_profile_dataset(
    profile_name=PROFILE_NAME,
    data_dir=DATA_DIR,
    market_filename=MARKET_FILENAME,
    output_filename=OUTPUT_FILENAME,
)

print(merged.shape)
merged.head()

montea: 35032 profile rows, 159836 market rows -> 35032 merged rows (100.0% of profile matched)
Saved to c:\Users\ThibaultBreemerschBn\ThibaultGitSandbox\thibault-sandbox\data\csvs\mergedmonteaANDda_belgium.csv
(35032, 7)


,dates,con,gen,off,inj,price [€/MWh],cleared_volume [MW]
0,2024-12-31 23:00:00+00:00,NaN,NaN,0.19325,0.0,10.62,742.525
1,2024-12-31 23:15:00+00:00,NaN,NaN,2.82500,0.0,10.62,742.525
2,2024-12-31 23:30:00+00:00,NaN,NaN,17.40000,0.0,10.62,742.525
3,2024-12-31 23:45:00+00:00,NaN,NaN,6.62500,0.0,10.62,742.525
4,2025-01-01 00:00:00+00:00,NaN,NaN,9.40000,0.0,10.27,695.600


## Sanity check

Quick visual check that the date range, row count, and columns look right before trusting the file for downstream notebooks (e.g. `optim_carmeuseV1.ipynb`).

In [3]:
print("Columns:", list(merged.columns))
print("Date range:", merged["dates"].min(), "->", merged["dates"].max())
print("Rows:", len(merged))
print("Any duplicate timestamps:", merged["dates"].duplicated().any())
print("Any missing quarter hours:", (merged["dates"].diff().dropna() != pd.Timedelta(minutes=15)).any())
merged.tail()
merged.head()

Columns: ['dates', 'con', 'gen', 'off', 'inj', 'price [€/MWh]', 'cleared_volume [MW]']
Date range: 2024-12-31 23:00:00+00:00 -> 2025-12-31 22:45:00+00:00
Rows: 35032
Any duplicate timestamps: False
Any missing quarter hours: True


,dates,con,gen,off,inj,price [€/MWh],cleared_volume [MW]
0,2024-12-31 23:00:00+00:00,NaN,NaN,0.19325,0.0,10.62,742.525
1,2024-12-31 23:15:00+00:00,NaN,NaN,2.82500,0.0,10.62,742.525
2,2024-12-31 23:30:00+00:00,NaN,NaN,17.40000,0.0,10.62,742.525
3,2024-12-31 23:45:00+00:00,NaN,NaN,6.62500,0.0,10.62,742.525
4,2025-01-01 00:00:00+00:00,NaN,NaN,9.40000,0.0,10.27,695.600


## Optional: rebuild every known profile at once

Set `RUN_ALL_PROFILES = True` and run to regenerate all merged CSVs in one go, e.g. after the market data or a source Excel file has been updated.

In [ ]:
RUN_ALL_PROFILES = False

if RUN_ALL_PROFILES:
    all_merged = {}
    for name in PROFILES:
        all_merged[name] = build_profile_dataset(
            profile_name=name,
            data_dir=DATA_DIR,
            market_filename=MARKET_FILENAME,
            output_filename=f"merged{name}ANDda_belgium.csv",
        )

carmeuse: 35136 profile rows, 159836 market rows -> 35136 merged rows (100.0% of profile matched)
Saved to c:\Users\ThibaultBreemerschBn\ThibaultGitSandbox\thibault-sandbox\data\csvs\mergedcarmeuseANDda_belgium.csv
montea: 35032 profile rows, 159836 market rows -> 35032 merged rows (100.0% of profile matched)
Saved to c:\Users\ThibaultBreemerschBn\ThibaultGitSandbox\thibault-sandbox\data\csvs\mergedmonteaANDda_belgium.csv
lemahieu: 35136 profile rows, 159836 market rows -> 35136 merged rows (100.0% of profile matched)
Saved to c:\Users\ThibaultBreemerschBn\ThibaultGitSandbox\thibault-sandbox\data\csvs\mergedlemahieuANDda_belgium.csv
